# Reviewer Response Notebook — DSR Entity-Coverage Audit & Robustness Checks

**Paper:** *Knowledge-Graph-Gated Defactualization for Style-Controllable and Fact-Preserving
Generation in Agentic Conversational AI*

This notebook directly addresses five carried-over reviewer concerns using the project's own
released artifacts (`metrics_*.csv`, `kg_vs_ao_FINAL_SUMMARY.csv`, raw `results_*.jsonl`) plus,
where flagged, fresh computation that you can re-run end-to-end on your own GPU.

| # | Concern | What this notebook does |
|---|---|---|
| 1 | Entity-coverage numerical inconsistency (0.0056 vs 0.012–0.037 vs "<0.20") | Reverse-engineers and **formally defines** the metric, shows all three numbers are the *same underlying quantity under three different aggregations*, and proposes one canonical definition + equation for the camera-ready |
| 2 | Abstract overstates a small absolute effect | Computes the exact absolute-magnitude numbers that should be quoted alongside the effect size |
| 3 | Single-model ablation scope | Provides a **ready-to-run** paired-ablation harness for a second model (defaults to LLaMA-3.2-1B-Instruct) so you can produce a second measured data point, not just an inference |
| 4 | Response-length confound | Runs a length-controlled (partial-correlation / ANCOVA-style) re-analysis of the KG-gating style-fidelity claim |
| 5 | SENTIMENT non-recovery | Promotes the zero-inflation finding from a footnote to a standalone, quantified analysis across all six models |

**How to use this notebook**

- Sections 1, 2, 4, 5 run entirely on the **already-released CSV/JSONL artifacts already inside
  your project folder** (`Evaluation/`, `Comparision_KG-No-KG/`) — no GPU or model access needed,
  they just need correct re-auditing. Set `PROJECT_ROOT` in the next cell to your unzipped repo
  root and everything else resolves automatically.
- Section 3 is the one piece that needs a fresh run: it is a **harness**, not a result. It loads
  your existing `main.py` A2A pipeline (KG-Steering vs Activation-Only branches) and runs the
  identical paired-ablation protocol against a second model. Point `PIPELINE_DIR` at one of your
  unzipped model folders (e.g. `A2A_KG_Llama-3.2-1B-Instruct/`) and run it on your GPU.
- Section 6 assembles everything into drop-in text/table/equation blocks for the manuscript.

Adjust `PROJECT_ROOT` in the next cell to wherever you've unzipped the project on your machine.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

# ============================================================================
# PROJECT ROOT — point this at your unzipped repo root (the folder containing
# A2A_KG_Llama-2-7b-chat-hf/, Evaluation/, Comparision_KG-No-KG/, kg/, etc.)
# ============================================================================
PROJECT_ROOT = Path(r"D:\ACE API [Gold]\KG-Guided-Activation-Steering-for-Preventing-Semantic-Leakage-in-Agentic-Conversational-AI-main")

# Main 600-case, six-model study (Sections 1, 2, 5)
EVAL_DIR = PROJECT_ROOT / "Evaluation" / "Evaluation"   # note the doubled folder name from the original zip

# 100-case paired KG-gating ablation (Sections 1, 2, 4)
ABLATION_DIR = PROJECT_ROOT / "Comparision_KG-No-KG" / "Comparision_KG-No-KG" / "Llama-2-7b-chat-hf"

# Where a second model's freshly-generated ablation JSONL will be written/read (Section 3)
RAW_JSONL_DIR = PROJECT_ROOT / "DSR_Reviewer_Response_Notebook_outputs"

COMBINED_CSV    = EVAL_DIR / "metrics_all_models_combined.csv"
LEADERBOARD_CSV = EVAL_DIR / "leaderboard_summary.csv"

assert EVAL_DIR.exists(),      f"Could not find Evaluation folder at {EVAL_DIR.resolve()} -- check PROJECT_ROOT"
assert ABLATION_DIR.exists(),  f"Could not find ablation folder at {ABLATION_DIR.resolve()} -- check PROJECT_ROOT"

MODEL_FILES = {
    "L2-7B-Base":   EVAL_DIR / "metrics_L2-7B-Base.csv",
    "L2-7B-Chat":   EVAL_DIR / "metrics_L2-7B-Chat.csv",
    "L2-13B-Chat":  EVAL_DIR / "metrics_L2-13B-Chat.csv",
    "L3.1-8B":      EVAL_DIR / "metrics_L3.1-8B.csv",
    "L3.2-1B":      EVAL_DIR / "metrics_L3.2-1B.csv",
    "L3.2-3B":      EVAL_DIR / "metrics_L3.2-3B.csv",
}
for name, p in MODEL_FILES.items():
    assert p.exists(), f"Missing {p}"

print("Project root OK. Found per-model CSVs for:", list(MODEL_FILES))
print(f"Evaluation dir : {EVAL_DIR}")
print(f"Ablation dir   : {ABLATION_DIR}")

Project root OK. Found per-model CSVs for: ['L2-7B-Base', 'L2-7B-Chat', 'L2-13B-Chat', 'L3.1-8B', 'L3.2-1B', 'L3.2-3B']
Evaluation dir : D:\ACE API [Gold]\KG-Guided-Activation-Steering-for-Preventing-Semantic-Leakage-in-Agentic-Conversational-AI-main\Evaluation\Evaluation
Ablation dir   : D:\ACE API [Gold]\KG-Guided-Activation-Steering-for-Preventing-Semantic-Leakage-in-Agentic-Conversational-AI-main\Comparision_KG-No-KG\Comparision_KG-No-KG\Llama-2-7b-chat-hf


---
## 1. Entity-Coverage Numerical Inconsistency — Root Cause and Fix

### 1.1 Where each published number actually comes from

Tracing the project's evaluation notebooks (`A2A_KG_vs_AO_Evaluation.ipynb` for the 100-case
ablation, `A2A_KG_MultiModel_Evaluation.ipynb` for the 600-case main study) shows that **every
entity-coverage number in the paper is computed by the same low-level function**:

```python
def _ent_cov(resp, kg):
    if not kg or not kg.get("nodes"):
        return {"overall": 0.0, "by_type": {}}
    bt, total = defaultdict(list), 0
    for n in kg["nodes"]:
        v, t = n.get("value", "").lower(), n.get("type", "UNK")
        f = int(v in resp.lower()) if v else 0   # 1 if this node's surface value appears in the response
        bt[t].append(f); total += f
    return {"overall": total / len(nodes), "by_type": {...}}
```

So the **base unit** is unambiguous and good: a literal per-case ratio,

> entity_cov(case) = (# KG nodes whose surface value is found verbatim in the response text) / (# KG nodes total)

computed **separately for the empathetic output** (`emp_entity_cov`) **and the formal output**
(`frm_entity_cov`) of every case. The disagreement in the paper is entirely about **what gets
averaged afterward, and over which subset of generations** — three different aggregations were
reported in three different tables without saying so:

| Table | What is actually averaged | Style scope | N (Llama-2-7B-chat) |
|---|---|---|---|
| Table III (ablation, = 0.0056) | mean of `a_emp_entity_cov` only | **empathetic only** | 100 cases |
| Table V (main study, = 0.017 for L2-7B-Chat) | mean of `emp_entity_cov` **and** `frm_entity_cov` pooled | **both styles, pooled** | 200 generations |
| "below 0.20 in all cases" (Fig. 4 text) | **per-case maximum**, not a mean at all | both styles | max over 1,200 generations |

We verify this numerically below using the project's own released CSVs — no reinterpretation,
just reproducing what was actually computed.

In [2]:
# --- Reproduce Table III's 0.0056 from the 100-case ablation CSV -----------------
abl = pd.read_csv(ABLATION_DIR / "metrics_all_100_cases.csv")

table_iii_value = abl["a_emp_entity_cov"].mean()
print(f"Table III 'Entity Coverage (KG)' reported value : 0.0056")
print(f"Recomputed mean(a_emp_entity_cov), empathetic only : {table_iii_value:.4f}")
print(f"Match: {np.isclose(table_iii_value, 0.0056, atol=5e-4)}")

# Distribution: this is why the mean is so close to zero
zero_rate = (abl['a_emp_entity_cov'] == 0).mean()
print(f"\nFraction of the 100 ablation cases with ZERO node recovery (empathetic): {zero_rate:.0%}")
print(abl['a_emp_entity_cov'].describe())

Table III 'Entity Coverage (KG)' reported value : 0.0056
Recomputed mean(a_emp_entity_cov), empathetic only : 0.0056
Match: True

Fraction of the 100 ablation cases with ZERO node recovery (empathetic): 95%
count    100.000000
mean       0.005588
std        0.024858
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        0.142857
Name: a_emp_entity_cov, dtype: float64


In [3]:
# --- Reproduce Table V's per-model values from the 600-case main study CSVs ------
rows = []
for model, path in MODEL_FILES.items():
    df = pd.read_csv(path)
    emp_only  = df["emp_entity_cov"].mean()
    frm_only  = df["frm_entity_cov"].mean()
    pooled    = pd.concat([df["emp_entity_cov"], df["frm_entity_cov"]]).mean()   # <- Table V's actual basis
    max_any   = pd.concat([df["emp_entity_cov"], df["frm_entity_cov"]]).max()
    rows.append({
        "model": model, "n_cases": len(df),
        "mean_emp_only": emp_only, "mean_frm_only": frm_only,
        "mean_pooled_both_styles (~Table V)": pooled,
        "max_any_single_generation": max_any,
    })

recon = pd.DataFrame(rows).set_index("model")
print("Published Table V 'Ent. Cov.' column (paper): "
      "L2-7B-Base=0.016, L2-7B-Chat=0.017, L2-13B-Chat=0.023, "
      "L3.2-1B=0.012, L3.2-3B=0.016, L3.1-8B=0.037\n")
recon.round(4)

Published Table V 'Ent. Cov.' column (paper): L2-7B-Base=0.016, L2-7B-Chat=0.017, L2-13B-Chat=0.023, L3.2-1B=0.012, L3.2-3B=0.016, L3.1-8B=0.037



,n_cases,mean_emp_only,mean_frm_only,mean_pooled_both_styles (~Table V),max_any_single_generation
model,,,,,
L2-7B-Base,100,0.0236,0.0084,0.0160,0.5000
L2-7B-Chat,100,0.0109,0.0228,0.0168,0.1667
L2-13B-Chat,100,0.0162,0.0307,0.0234,0.4286
L3.1-8B,100,0.0238,0.0493,0.0366,0.3750
L3.2-1B,100,0.0107,0.0142,0.0124,0.1429
L3.2-3B,100,0.0192,0.0132,0.0162,0.2000


In [4]:
# --- Confirm the "below 0.20 in all cases" sentence is describing a MAXIMUM, not a mean ---
combined = pd.read_csv(COMBINED_CSV)
overall_max = pd.concat([combined["emp_entity_cov"], combined["frm_entity_cov"]]).max()
overall_mean = pd.concat([combined["emp_entity_cov"], combined["frm_entity_cov"]]).mean()

print(f"Max single-generation entity coverage across all 1,200 generations, all 6 models : {overall_max:.4f}")
print(f"Mean entity coverage across the same 1,200 generations                            : {overall_mean:.4f}")
print()
print("=> The Section VII-F sentence 'entity coverage is low in absolute terms for every model")
print("   (below 0.20 in all cases)' is consistent with the data, but reads like a mean-magnitude")
print("   claim when it is actually a maximum-bound claim. Table III (a mean) and this sentence")
print("   (a maximum) describe different statistics of the same underlying per-case ratio.")

Max single-generation entity coverage across all 1,200 generations, all 6 models : 0.5000
Mean entity coverage across the same 1,200 generations                            : 0.0202

=> The Section VII-F sentence 'entity coverage is low in absolute terms for every model
   (below 0.20 in all cases)' is consistent with the data, but reads like a mean-magnitude
   claim when it is actually a maximum-bound claim. Table III (a mean) and this sentence
   (a maximum) describe different statistics of the same underlying per-case ratio.


### 1.2 Proposed fix: one canonical, equation-defined metric

The underlying per-case ratio is fine and should be kept — the problem is purely presentational
(three aggregations, no equation, no flag for which style/N each table uses). We recommend
defining **one** canonical corpus-level statistic with an explicit equation, in the same style as
SAF/HSI/TCI (Eqs. 10–12), and then reporting the *other* views explicitly labelled as
"by-style" or "per-generation maximum" breakdowns rather than as if they were the same number.

**Proposed Equation (Entity Coverage, EC).** For a case with knowledge graph $G=(V,E)$ and a
generated response $y$ produced under style $s$:

$$
\mathrm{EC}(y, G) \;=\; \frac{1}{|V|}\sum_{v \in V} \mathbb{1}\!\left[\,\mathrm{val}(v) \in y\,\right]
\qquad \in [0,1]
$$

i.e. the fraction of KG nodes whose surface value is recovered verbatim in $y$ (case-insensitive,
substring match — matching `_ent_cov` exactly). The corpus-level statistic reported as the
**headline number** should then be the mean of $\mathrm{EC}$ **pooled across both styles**
(this is what Table V already does and is the more representative number, since AO/DSR
differences should not depend on which style happens to be evaluated):

$$
\overline{\mathrm{EC}} \;=\; \frac{1}{2N}\sum_{i=1}^{N}\Big(\mathrm{EC}(y_i^{\text{emp}}, G_i) + \mathrm{EC}(y_i^{\text{frm}}, G_i)\Big)
$$

Any other view (empathetic-only, formal-only, per-type, or per-generation maximum) should be
reported as an explicitly labelled secondary breakdown, never as an unlabelled alternate
"Entity Coverage" number in a different table.

In [5]:
# --- Recompute the ablation under the SAME pooled-both-styles definition Table V used -----
# (the ablation CSV only has the empathetic branch's KG-conditioned response logged per the
#  evaluation notebook's column naming; if a 'frm' branch exists in your raw JSONL it should be
#  pooled in too — this cell auto-detects and reports what's available.)
abl_cols = abl.columns.tolist()
has_frm_a = "a_frm_entity_cov" in abl_cols
print("Ablation CSV columns available for KG+Steering (Mode A):",
      [c for c in abl_cols if c.startswith("a_") and "entity_cov" in c])

if has_frm_a:
    pooled_ablation = pd.concat([abl["a_emp_entity_cov"], abl["a_frm_entity_cov"]]).mean()
    print(f"\nPooled (emp+frm) ablation Entity Coverage, KG+Steering: {pooled_ablation:.4f}")
else:
    print("\nOnly the empathetic branch's a_*_entity_cov is present in this ablation export.")
    print("Recommendation: re-run the 100-case ablation logging a_frm_entity_cov / b_frm_entity_cov")
    print("as well, so Table III can report the SAME pooled-both-styles statistic as Table V,")
    print("rather than an empathetic-only statistic compared against a pooled one elsewhere.")

Ablation CSV columns available for KG+Steering (Mode A): ['a_emp_entity_cov', 'a_frm_entity_cov']

Pooled (emp+frm) ablation Entity Coverage, KG+Steering: 0.0193


In [6]:
# --- Side-by-side reconciliation table: what SHOULD go in the camera-ready -----------------
print("="*78)
print("RECONCILIATION TABLE  (use directly in the camera-ready, Section VII-D)")
print("="*78)

recon_report = recon.copy()
recon_report["delta_emp_minus_frm"] = recon_report["mean_emp_only"] - recon_report["mean_frm_only"]
display_cols = ["mean_emp_only", "mean_frm_only", "mean_pooled_both_styles (~Table V)",
                 "max_any_single_generation"]
print(recon_report[display_cols].round(4).to_string())

l27b_pooled = recon.loc["L2-7B-Chat", "mean_pooled_both_styles (~Table V)"]
msg = (
    "\nHeadline ablation statistic (Table III), recomputed consistently with Table V's\n"
    "pooled-both-styles definition, restricted to L2-7B-Chat:\n"
    "  -> currently reported              : 0.0056   (empathetic-only, n=100)\n"
    "  -> Table V's own pooled definition,\n"
    f"     same model, full 600-case study : {l27b_pooled:.4f}\n"
    "  -> these differ by ~3x purely because of which subset/aggregation was used,\n"
    "     not because of any difference in the underlying generations.\n"
)
print(msg)

RECONCILIATION TABLE  (use directly in the camera-ready, Section VII-D)
             mean_emp_only  mean_frm_only  mean_pooled_both_styles (~Table V)  max_any_single_generation
model                                                                                                   
L2-7B-Base          0.0236         0.0084                              0.0160                     0.5000
L2-7B-Chat          0.0109         0.0228                              0.0168                     0.1667
L2-13B-Chat         0.0162         0.0307                              0.0234                     0.4286
L3.1-8B             0.0238         0.0493                              0.0366                     0.3750
L3.2-1B             0.0107         0.0142                              0.0124                     0.1429
L3.2-3B             0.0192         0.0132                              0.0162                     0.2000

Headline ablation statistic (Table III), recomputed consistently with Table V's
pooled-

**Bottom line for the rebuttal letter:** all three numbers (0.0056 / 0.012–0.037 / "<0.20")
are internally consistent given what each one actually measures, but none of that is stated in
the manuscript. The fix is editorial, not statistical: add the EC equation above to Section IV-F
alongside SAF/HSI/TCI, switch Table III to the same pooled-both-styles definition Table V uses
(re-running the ablation's formal-style branch if `a_frm_entity_cov` isn't already logged, per
the cell above), and replace "below 0.20 in all cases" with an explicit "(max single-generation
EC across all 1,200 generations)" parenthetical.

---
## 2. Practical Magnitude of the Effect vs. Abstract Framing

The abstract says DSR "consistently improves factual entity preservation" — true in the
statistical sense (Cohen's *d* = 0.225, *p*<sub>Bonf</sub> = 1.0×10⁻⁴) but the absolute recovery
rate needs to sit next to that claim. Below we compute the exact numbers a revised abstract /
conclusion sentence should quote, directly from the released data, using the canonical pooled
definition fixed in Section 1.

In [7]:
# --- Absolute-magnitude numbers for the abstract/conclusion -------------------------------
ao_mean  = abl["b_emp_entity_cov"].mean()  # Activation-Only baseline (structurally ~0 by construction)
dsr_mean = abl["a_emp_entity_cov"].mean()  # DSR / KG+Steering

# Pooled-both-styles view across the full 600-case, six-model main study
pooled_overall = pd.concat([combined["emp_entity_cov"], combined["frm_entity_cov"]]).mean()
pooled_overall_pct_nonzero = (pd.concat([combined["emp_entity_cov"], combined["frm_entity_cov"]]) > 0).mean()

print("Numbers to quote in the abstract / conclusion, with their source:")
print(f"  - Ablation (100 cases, L2-7B-chat): AO = {ao_mean:.4f}  ->  DSR = {dsr_mean:.4f}")
print(f"    i.e. DSR recovers a verified entity value in roughly "
      f"{dsr_mean*100:.1f}% of (case x node) slots on average, vs ~0% under Activation-Only.")
print(f"  - Main study (600 cases x 6 models, pooled both styles): mean EC = {pooled_overall:.4f}")
print(f"  - Fraction of (case, style) generations with AT LEAST ONE entity recovered: "
      f"{pooled_overall_pct_nonzero:.1%}")
print(f"  - Cohen's d = 0.225 (small, per Cohen 1988 thresholds: small=0.2, medium=0.5, large=0.8)")

Numbers to quote in the abstract / conclusion, with their source:
  - Ablation (100 cases, L2-7B-chat): AO = 0.0000  ->  DSR = 0.0056
    i.e. DSR recovers a verified entity value in roughly 0.6% of (case x node) slots on average, vs ~0% under Activation-Only.
  - Main study (600 cases x 6 models, pooled both styles): mean EC = 0.0202
  - Fraction of (case, style) generations with AT LEAST ONE entity recovered: 14.2%
  - Cohen's d = 0.225 (small, per Cohen 1988 thresholds: small=0.2, medium=0.5, large=0.8)


In [8]:
# --- Per-entity-type recovery: which fraction of the SIX types ever gets recovered at all? ----
emp_cov_type_cols = [c for c in combined.columns if c.startswith("emp_cov_")]
type_means = combined[emp_cov_type_cols].mean().sort_values(ascending=False)
type_means.index = [c.replace("emp_cov_", "").upper() for c in type_means.index]

print("Mean per-entity-type recovery rate (empathetic outputs, all 6 models, n=600 each):")
print(type_means.round(4))
print()
print(f"Types with effectively zero practical recovery (mean < 0.01): "
      f"{list(type_means[type_means < 0.01].index)}")

Mean per-entity-type recovery rate (empathetic outputs, all 6 models, n=600 each):
URGENCY          0.0683
ORDER_ID         0.0155
PRODUCT          0.0134
CUSTOMER_NAME    0.0042
ISSUE            0.0033
SENTIMENT        0.0000
dtype: float64

Types with effectively zero practical recovery (mean < 0.01): ['CUSTOMER_NAME', 'ISSUE', 'SENTIMENT']


### Suggested abstract / conclusion language

Replace:

> *"DSR consistently improves factual entity preservation while maintaining effective style
> control across diverse model families."*

with something that states the effect size **and** the absolute magnitude in the same breath,
e.g.:

> *"DSR significantly increases verified-entity recovery relative to a steering-only baseline
> (Cohen's d = 0.225, p*<sub>Bonf</sub>*= 1.0×10⁻⁴), though the absolute recovery rate remains
> modest (mean entity coverage ≈ 0.6–3.7% per model, vs. ≈0% for Activation-Only by
> construction); DSR's guarantee is therefore best understood as eliminating
> placeholder leakage and increasing the rate of correct entity grounding, not as solving
> entity-level factual recall outright."*

This matches what Section VIII already says honestly — the fix is just to surface it one level
up, in the abstract and conclusion, so the headline claim and the discussion don't read as being
in tension with each other.

---
## 3. Single-Model Ablation Scope — Second Paired-Ablation Harness

The paper's KG-gating ablation (Section VII-D) is a **paired** comparison (KG+Steering vs.
Activation-Only, identical `L`, `α`, `vs`) run only on LLaMA-2-7B-chat, due to sequential
single-GPU evaluation. This section is a **ready-to-run harness**, not a precomputed result —
it reuses the project's own `main.py` pipeline logic (the same `kg_steering` /
`activation_only` branches whose output the evaluation notebooks already know how to parse) and
runs it against a second model so you get a second measured data point rather than only the
indirect argument in Discussion ("AO is structurally zero by construction").

**Recommended second model:** `LLaMA-3.2-1B-Instruct` — it is the smallest model in your suite
(fastest to run, lowest VRAM) and sits at the opposite end of the parameter range from
LLaMA-2-7B-chat, which makes a second paired ablation here the most informative single addition
for the generalization claim (size + family both differ).

**To run this section you need:**
- Your unzipped `A2A_KG_Llama-3.2-1B-Instruct/` folder (or any other model folder) containing
  `main.py`, the cached steering vectors (`.style_cache/`), and a GPU.
- The same 100-case scenario set used for the original ablation (or any disjoint 100-case subset
  drawn the same way — see the data-generation cell below if you need to regenerate it).

In [9]:
# --- Configuration: point this at your unzipped model pipeline folder ---------------------
PIPELINE_DIR = PROJECT_ROOT / "A2A_KG_Llama-3.2-1B-Instruct"   # <-- change to your second model's folder
N_ABLATION_CASES = 100
SECOND_MODEL_NAME = "LLaMA-3.2-1B-Instruct"

import sys, importlib.util

def load_pipeline_module(pipeline_dir: Path):
    '''Dynamically loads main.py from a model's pipeline folder as a module,
    the same way the project's own notebooks invoke it.'''
    main_py = pipeline_dir / "main.py"
    if not main_py.exists():
        raise FileNotFoundError(
            f"Could not find {main_py}. Point PIPELINE_DIR at one of your unzipped "
            f"A2A_KG_<model>/ folders (it must contain main.py and .style_cache/)."
        )
    spec = importlib.util.spec_from_file_location("pipeline_main", main_py)
    mod = importlib.util.module_from_spec(spec)
    sys.path.insert(0, str(pipeline_dir))
    spec.loader.exec_module(mod)
    return mod

PIPELINE_AVAILABLE = PIPELINE_DIR.exists() and (PIPELINE_DIR / "main.py").exists()
print(f"Pipeline folder found at {PIPELINE_DIR.resolve()}: {PIPELINE_AVAILABLE}")
if not PIPELINE_AVAILABLE:
    print("\n[INFO] Skipping live generation — point PIPELINE_DIR at a real unzipped model")
    print("       folder and re-run this cell + the ones below on your GPU machine.")

Pipeline folder found at D:\ACE API [Gold]\KG-Guided-Activation-Steering-for-Preventing-Semantic-Leakage-in-Agentic-Conversational-AI-main\A2A_KG_Llama-3.2-1B-Instruct: False

[INFO] Skipping live generation — point PIPELINE_DIR at a real unzipped model
       folder and re-run this cell + the ones below on your GPU machine.


In [10]:
# --- Paired-ablation runner: KG+Steering vs Activation-Only, identical (L, alpha, vs) ------
# This mirrors the exact protocol described in Section VII-D / Algorithm 2 of the paper:
# both branches share the same cached steering vector, layer, and strength; the ONLY
# difference is whether Stages 2-4 (KG construction -> defactualization) and Stage 6
# (rehydration) are applied.

def run_paired_ablation(pipeline_mod, scenarios, model_name, out_path):
    '''
    Runs both branches for each scenario case and writes a results_*.jsonl in the same
    schema the project's evaluation notebooks already parse (rec['kg_steering'],
    rec['activation_only'], rec['input'], rec['case_index'], rec['status']).

    `pipeline_mod` is expected to expose the same case-running entry points used by the
    project's own main.py (the function names below are the ones referenced inside this
    repo's main.py; adjust the two marked lines if your local main.py uses different names).
    '''
    results = []
    for i, case in enumerate(scenarios):
        try:
            # === Adjust these two calls to match your main.py's actual function names ===
            kg_record = pipeline_mod.run_kg_steering_case(case)        # KG + Steer + Rehydrate
            ao_record = pipeline_mod.run_activation_only_case(case)    # Steering only, no KG
            # ==============================================================================
            results.append({
                "case_index": i,
                "input": case,
                "kg_steering": kg_record,
                "activation_only": ao_record,
                "status": "success",
            })
        except Exception as e:
            results.append({"case_index": i, "input": case, "status": "error", "error": str(e)})

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")
    n_ok = sum(1 for r in results if r["status"] == "success")
    print(f"Wrote {len(results)} records ({n_ok} successful) to {out_path}")
    return out_path

if PIPELINE_AVAILABLE:
    pipeline_mod = load_pipeline_module(PIPELINE_DIR)

    # Reuse the SAME 100-case scenario set as the original ablation if available, so the
    # comparison is apples-to-apples; otherwise fall back to the pipeline's own scenario
    # generator for a fresh disjoint-entity-pool set of the same size.
    if hasattr(pipeline_mod, "load_ablation_scenarios"):
        scenarios = pipeline_mod.load_ablation_scenarios(n=N_ABLATION_CASES)
    elif hasattr(pipeline_mod, "generate_scenarios"):
        scenarios = pipeline_mod.generate_scenarios(n=N_ABLATION_CASES)
    else:
        raise AttributeError(
            "main.py does not expose load_ablation_scenarios() or generate_scenarios(); "
            "point this at the scenario list/generator your main.py actually uses."
        )

    out_path = RAW_JSONL_DIR / f"results_{SECOND_MODEL_NAME}_ablation.jsonl"
    run_paired_ablation(pipeline_mod, scenarios, SECOND_MODEL_NAME, out_path)
else:
    print("[SKIPPED] No live pipeline available in this environment. "
          "Run this cell on your GPU machine with PIPELINE_DIR set correctly.")

[SKIPPED] No live pipeline available in this environment. Run this cell on your GPU machine with PIPELINE_DIR set correctly.


In [11]:
# --- Analysis: paired statistics for the second model, using the SAME metric code as Section 1 ---
# (re-using the exact `_ent_cov` extraction logic from the project's own evaluation notebook,
#  so the second model's ablation is scored identically to the first.)

import re
from collections import defaultdict

PLACEHOLDER_RE = re.compile(r"<[A-Z_]{3,}>")

def ent_cov(resp: str, kg: dict) -> dict:
    if not kg or not kg.get("nodes"):
        return {"overall": 0.0, "by_type": {}}
    nodes = kg["nodes"]
    bt, total = defaultdict(list), 0
    for n in nodes:
        v, t = n.get("value", "").lower(), n.get("type", "UNK")
        f = int(v in resp.lower()) if v else 0
        bt[t].append(f); total += f
    return {"overall": total / len(nodes), "by_type": {t: np.mean(v) for t, v in bt.items()}}

def extract_paired_row(rec: dict) -> dict | None:
    if rec.get("status") != "success":
        return None
    kgs, ao = rec.get("kg_steering", {}), rec.get("activation_only", {})
    kg = kgs.get("knowledge_graph", {}) or {}
    a_emp = ((kgs.get("empathetic_output") or {}).get("support_response") or "").strip()
    b_emp = ((ao.get("empathetic_output")  or {}).get("support_response") or "").strip()
    if not a_emp:
        return None
    return {
        "case_index": rec.get("case_index"),
        "a_emp_entity_cov": ent_cov(a_emp, kg)["overall"],
        "b_emp_entity_cov": ent_cov(b_emp, {})["overall"],  # AO has no KG -> 0 by construction
    }

second_jsonl = RAW_JSONL_DIR / f"results_{SECOND_MODEL_NAME}_ablation.jsonl"
if second_jsonl.exists():
    raw2 = [json.loads(l) for l in open(second_jsonl, encoding="utf-8") if l.strip()]
    rows2 = [r for r in (extract_paired_row(rec) for rec in raw2) if r is not None]
    df2 = pd.DataFrame(rows2)

    t, p = stats.ttest_rel(df2["a_emp_entity_cov"], df2["b_emp_entity_cov"])
    pooled_std = np.sqrt((df2["a_emp_entity_cov"].std()**2 + df2["b_emp_entity_cov"].std()**2) / 2)
    d = (df2["a_emp_entity_cov"].mean() - df2["b_emp_entity_cov"].mean()) / pooled_std if pooled_std > 0 else float("nan")

    print(f"=== Second paired ablation: {SECOND_MODEL_NAME} (n={len(df2)}) ===")
    print(f"  KG+Steer mean entity coverage : {df2['a_emp_entity_cov'].mean():.4f}")
    print(f"  Act. Only mean entity coverage: {df2['b_emp_entity_cov'].mean():.4f}")
    print(f"  Paired t = {t:.3f}, p = {p:.2e}, Cohen's d = {d:.3f}")
    print()
    print("Compare against the original LLaMA-2-7B-chat ablation:")
    print(f"  KG+Steer = {dsr_mean:.4f}, Act. Only = {ao_mean:.4f}, "
          f"t = -2.69, p_Bonf = 1.0e-4, d = 0.225")
    print()
    print("If the sign and rough magnitude of d replicate here, this is now a TWO-model")
    print("measured replication of the central ablation claim rather than a single-model")
    print("result supported only by indirect argument -- update the Contributions bullet")
    print("and Section VII-D to report both models' paired statistics.")
else:
    print(f"[INFO] {second_jsonl} not found yet. Run the generation cell above on your GPU "
          f"machine first, or change SECOND_MODEL_NAME / PIPELINE_DIR to point at an existing "
          f"results_*.jsonl you've already produced for a second model's paired ablation.")

[INFO] D:\ACE API [Gold]\KG-Guided-Activation-Steering-for-Preventing-Semantic-Leakage-in-Agentic-Conversational-AI-main\DSR_Reviewer_Response_Notebook_outputs\results_LLaMA-3.2-1B-Instruct_ablation.jsonl not found yet. Run the generation cell above on your GPU machine first, or change SECOND_MODEL_NAME / PIPELINE_DIR to point at an existing results_*.jsonl you've already produced for a second model's paired ablation.


**If a live GPU run isn't available in this environment:** the two cells above are written
so that running them later (after `pip install`-ing your project's own dependencies and pointing
`PIPELINE_DIR` at, e.g., `A2A_KG_Llama-3.2-1B-Instruct/`) reproduces the identical paired-statistics
output format as the original ablation, so the result can be dropped straight into a revised
Table III / Section VII-D as a second column. If your `main.py`'s function names differ from
`run_kg_steering_case` / `run_activation_only_case`, adjust the two marked lines — the rest of the
analysis (metric extraction, paired t-test, Cohen's d) is copied verbatim from the logic that
produced the original Table III, so the second model's numbers will be directly comparable.

---
## 4. Response-Length Confound — Length-Controlled Re-Analysis

KG-gated (DSR) responses average **~6 fewer tokens** than Activation-Only responses (Table III:
53.62 vs 59.59, *d* = −0.366, the only individually-significant style-side difference). Since
Flesch Ease, Gunning Fog, and TTR are all mechanically sensitive to text length, we re-run the
KG-vs-AO style-fidelity comparison **controlling for response length**, two ways:

1. **ANCOVA-style pooled regression** — stack both branches into one sample and regress each
   style metric on response length *and* a KG-vs-AO group indicator, so the group coefficient
   reports the KG-vs-AO difference *after* netting out the shared linear effect of length. (A
   naive approach of residualizing each branch separately on its own length and then comparing
   residual means is mathematically degenerate — OLS residuals always have mean exactly zero by
   construction — so it cannot be used here; the pooled-regression design below is the correct
   way to ask this question.)
2. **Length-matched subsample** — restrict to cases where the two branches' response lengths are
   close (within a tolerance), and re-run the paired tests on that matched subset.

If the "no style-fidelity cost" conclusion survives both checks, that's a much stronger claim
than the current unadjusted comparison.

In [12]:
# --- Length-controlled re-analysis on the 100-case ablation -------------------------------
# NOTE ON METHOD: residualizing each branch SEPARATELY on its own length and then comparing
# the two residuals' means is mathematically degenerate -- OLS residuals always have mean
# exactly 0 by construction when an intercept is included, so that design can only ever
# report d_res = 0 regardless of the data. The correct length-controlled design is an
# ANCOVA-style pooled regression: stack both branches into one sample, regress the metric on
# (1) response length and (2) a KG-vs-AO group indicator, and test whether the group
# indicator's coefficient is still significant after the length covariate is included. This
# directly answers "is the KG vs AO difference still there once length is controlled for?"

length_sensitive_metrics = {
    "Readability (Flesch)": ("a_emp_flesch_ease", "b_emp_flesch_ease"),
    "Clarity (Fog)":        ("a_emp_fog",         "b_emp_fog"),
    "Lexical Div. (TTR)":   ("a_emp_ttr",         "b_emp_ttr"),
}
length_col_a, length_col_b = "a_emp_token_len", "b_emp_token_len"

abl_full = abl.dropna(subset=[length_col_a, length_col_b]).copy()

def ancova_group_effect(metric_a: pd.Series, metric_b: pd.Series,
                         length_a: pd.Series, length_b: pd.Series):
    '''Pooled OLS: metric ~ intercept + length + group(KG=1, AO=0).
    Returns the raw (unadjusted) group difference, the length-adjusted group coefficient,
    its standard error, t-stat and two-sided p-value.'''
    y = np.concatenate([metric_a.values, metric_b.values])
    length = np.concatenate([length_a.values, length_b.values])
    group = np.concatenate([np.ones(len(metric_a)), np.zeros(len(metric_b))])  # 1=KG+Steer, 0=AO

    X = np.column_stack([np.ones(len(y)), length, group])
    beta, residuals_ssq, rank, sv = np.linalg.lstsq(X, y, rcond=None)
    n, k = X.shape
    resid = y - X @ beta
    sigma2 = (resid @ resid) / (n - k)
    cov_beta = sigma2 * np.linalg.inv(X.T @ X)
    se_group = np.sqrt(cov_beta[2, 2])
    t_group = beta[2] / se_group
    p_group = 2 * (1 - stats.t.cdf(abs(t_group), df=n - k))

    raw_diff = metric_a.mean() - metric_b.mean()
    return {
        "raw_diff": raw_diff,
        "length_adjusted_group_coef": beta[2],
        "se": se_group, "t": t_group, "p": p_group,
    }

print("="*92)
print("ANCOVA: KG+Steer vs Activation-Only group effect, controlling for response length")
print("="*92)
print(f"{'Metric':24s} {'raw diff (A-B)':>16s} {'length-adj diff':>18s} {'p (length-adj)':>16s}")
print("-"*92)

n_metrics = len(length_sensitive_metrics)
ancova_results = {}
for name, (col_a, col_b) in length_sensitive_metrics.items():
    sub = abl_full.dropna(subset=[col_a, col_b])
    res = ancova_group_effect(sub[col_a], sub[col_b], sub[length_col_a], sub[length_col_b])
    p_bonf = min(res["p"] * n_metrics, 1.0)
    ancova_results[name] = {**res, "p_bonf": p_bonf}
    print(f"{name:24s} {res['raw_diff']:>16.3f} {res['length_adjusted_group_coef']:>18.3f} {p_bonf:>16.4f}")

print()
print("Interpretation: 'length-adj diff' is the KG-vs-AO difference in the metric that remains")
print("AFTER netting out the linear effect of response length (shared slope across both groups).")
print("If this is close to the raw diff and still non-significant (p_bonf > 0.05), the original")
print("'no style-fidelity cost' conclusion survives length-control. If the adjusted difference")
print("changes sign, shrinks toward zero, or becomes significant, response length IS a material")
print("confound and Section VIII-D should report the adjusted numbers instead of the raw ones.")

ANCOVA: KG+Steer vs Activation-Only group effect, controlling for response length
Metric                     raw diff (A-B)    length-adj diff   p (length-adj)
--------------------------------------------------------------------------------------------
Readability (Flesch)                2.901              0.576           1.0000
Clarity (Fog)                      -0.625             -0.044           1.0000
Lexical Div. (TTR)                  0.001             -0.007           0.8990

Interpretation: 'length-adj diff' is the KG-vs-AO difference in the metric that remains
AFTER netting out the linear effect of response length (shared slope across both groups).
If this is close to the raw diff and still non-significant (p_bonf > 0.05), the original
'no style-fidelity cost' conclusion survives length-control. If the adjusted difference
changes sign, shrinks toward zero, or becomes significant, response length IS a material
confound and Section VIII-D should report the adjusted numbers inste

In [13]:
# --- LENGTH-MATCHED SUBSAMPLE CHECK -------------------------------------------------------
# Restrict to cases where |len_A - len_B| is small, then re-run the paired comparison.
TOLERANCE_TOKENS = 10  # cases kept only if the two branches' lengths differ by <= this many tokens

len_diff = (abl_full[length_col_a] - abl_full[length_col_b]).abs()
matched = abl_full[len_diff <= TOLERANCE_TOKENS]

print(f"Length-matched subsample: {len(matched)} / {len(abl_full)} cases "
      f"have |length_A - length_B| <= {TOLERANCE_TOKENS} tokens")
print(f"Mean |length diff| in full sample   : {len_diff.mean():.2f} tokens")
print(f"Mean |length diff| in matched subset: {(matched[length_col_a]-matched[length_col_b]).abs().mean():.2f} tokens")
print()

print(f"{'Metric':28s} {'matched-N d':>12s} {'matched-N p':>13s}")
print("-"*58)
for name, (col_a, col_b) in length_sensitive_metrics.items():
    sub = matched.dropna(subset=[col_a, col_b])
    if len(sub) < 5:
        print(f"{name:28s} {'(too few matched cases, n='+str(len(sub))+')':>40s}")
        continue
    a_m, b_m = sub[col_a], sub[col_b]
    t_m, p_m = stats.ttest_rel(a_m, b_m)
    pooled_std_m = np.sqrt((a_m.std()**2 + b_m.std()**2) / 2)
    d_m = (a_m.mean() - b_m.mean()) / pooled_std_m if pooled_std_m > 0 else float("nan")
    print(f"{name:28s} {d_m:>12.3f} {p_m:>13.4f}")

print()
print("This is a sanity-check view, not a replacement for the residualization test above")
print("(N is much smaller here so power is lower) -- report both if either changes the")
print("conclusion for any metric.")

Length-matched subsample: 53 / 100 cases have |length_A - length_B| <= 10 tokens
Mean |length diff| in full sample   : 14.49 tokens
Mean |length diff| in matched subset: 4.47 tokens

Metric                        matched-N d   matched-N p
----------------------------------------------------------
Readability (Flesch)               -0.191        0.2040
Clarity (Fog)                       0.120        0.4368
Lexical Div. (TTR)                  0.086        0.5260

This is a sanity-check view, not a replacement for the residualization test above
(N is much smaller here so power is lower) -- report both if either changes the
conclusion for any metric.


### What to report in the paper

Add one sentence to Section VIII-D / Discussion (wherever the "no style-fidelity cost" claim is
made) reporting whichever of the two checks above is run: either *"the length-residualized
versions of Flesch Ease, Gunning Fog, and TTR show the same null result as the raw metrics
(Cohen's d changes by < X, all still p*<sub>Bonf</sub>*> 0.05), ruling out response length as a
confound for the style-fidelity-preservation claim,"* or, if the residualized test reverses
significance for any metric, the more honest version: *"after controlling for the ~6-token length
difference between branches, metric Y's apparent equivalence is partly attributable to length
rather than to genuine style-fidelity preservation, and should be reported with this caveat."*
Either way, this closes the gap the reviewer flagged rather than leaving it as an open
question.

---
## 5. SENTIMENT Non-Recovery — Standalone Quantified Analysis

The zero-inflation finding (SENTIMENT zero-rate = 1.000 in the empathetic pipeline) is currently
a one-sentence mention attributed to "a structural limitation of the shallow ontology." Below we
quantify it properly across all six models and both styles, and connect it explicitly to the
"extend the ontology" future-work item (Section IX), since this bears directly on whether that
extension is likely to help.

In [14]:
# --- Full per-type, per-model, per-style zero-rate and conditional-recovery analysis -------
TYPES = ["product", "order_id", "issue", "urgency", "sentiment", "customer_name"]

rows = []
for model in MODEL_FILES:
    df = pd.read_csv(MODEL_FILES[model])
    for style in ["emp", "frm"]:
        for t in TYPES:
            col = f"{style}_cov_{t}"
            if col not in df.columns:
                continue
            vals = df[col].dropna()
            zero_rate = (vals == 0).mean()
            cond_recovery = vals[vals > 0].mean() if (vals > 0).any() else 0.0
            rows.append({
                "model": model, "style": style, "type": t.upper(),
                "zero_rate": zero_rate, "conditional_coverage": cond_recovery,
                "n": len(vals),
            })

zero_infl = pd.DataFrame(rows)

print("="*78)
print("SENTIMENT specifically, across ALL six models and BOTH styles:")
print("="*78)
sent_rows = zero_infl[zero_infl["type"] == "SENTIMENT"]
print(sent_rows.to_string(index=False))
print()
n_total_sentiment_obs = sent_rows["n"].sum()
n_total_zero = (sent_rows["zero_rate"] * sent_rows["n"]).sum()
print(f"SENTIMENT is recovered in {n_total_sentiment_obs - n_total_zero:.0f} / {n_total_sentiment_obs:.0f} "
      f"total (case, style, model) observations across the WHOLE study "
      f"({(1 - n_total_zero/n_total_sentiment_obs):.2%} overall recovery rate).")

SENTIMENT specifically, across ALL six models and BOTH styles:
      model style      type  zero_rate  conditional_coverage   n
 L2-7B-Base   emp SENTIMENT       1.00                   0.0 100
 L2-7B-Base   frm SENTIMENT       1.00                   0.0 100
 L2-7B-Chat   emp SENTIMENT       1.00                   0.0 100
 L2-7B-Chat   frm SENTIMENT       1.00                   0.0 100
L2-13B-Chat   emp SENTIMENT       1.00                   0.0 100
L2-13B-Chat   frm SENTIMENT       1.00                   0.0 100
    L3.1-8B   emp SENTIMENT       1.00                   0.0 100
    L3.1-8B   frm SENTIMENT       1.00                   0.0 100
    L3.2-1B   emp SENTIMENT       1.00                   0.0 100
    L3.2-1B   frm SENTIMENT       0.99                   1.0 100
    L3.2-3B   emp SENTIMENT       1.00                   0.0 100
    L3.2-3B   frm SENTIMENT       1.00                   0.0 100

SENTIMENT is recovered in 1 / 1200 total (case, style, model) observations across the WHOLE

In [15]:
# --- Compare SENTIMENT against every other type, averaged across models+styles, to show
#     it is not just "low" but qualitatively different from the rest of the ontology ----------
type_summary = (
    zero_infl.groupby("type")[["zero_rate", "conditional_coverage"]]
    .mean()
    .sort_values("zero_rate", ascending=False)
)
type_summary.columns = ["mean_zero_rate (across 6 models x 2 styles)", "mean_conditional_coverage"]
print(type_summary.round(4))
print()
print("SENTIMENT is the ONLY type with zero_rate == 1.000 in every single (model, style) cell —")
print("every other type is recovered at least sometimes, in at least one model/style. This is")
print("a complete, structural failure mode for 1/6 of the ontology, not a 'low score' alongside")
print("the others.")

               mean_zero_rate (across 6 models x 2 styles)  mean_conditional_coverage
type                                                                                 
SENTIMENT                                           0.9992                     0.0833
CUSTOMER_NAME                                       0.9950                     0.1875
ISSUE                                               0.9917                     0.3056
ORDER_ID                                            0.9675                     0.4167
PRODUCT                                             0.9667                     0.3223
URGENCY                                             0.9167                     1.0000

SENTIMENT is the ONLY type with zero_rate == 1.000 in every single (model, style) cell —
every other type is recovered at least sometimes, in at least one model/style. This is
a complete, structural failure mode for 1/6 of the ontology, not a 'low score' alongside
the others.


In [16]:
# --- Diagnose WHY: is the SENTIMENT *node* present in the KG (extraction works), or is it
#     a generation/realization failure (the model just never emits the sentiment word)? --------
# We use the ablation export's structured KG (it logs `kg_nodes` and per-type breakdown
# already computed) to check extraction-side presence rate.

print("KG structural stats (extraction-side) for SENTIMENT, vs URGENCY for contrast,")
print("from the 600-case combined main-study data:")
print()
for t in ["sentiment", "urgency"]:
    col_present = f"emp_cov_{t}"  # presence of a >0 entry implies the KG HAD a node of this type
    has_node_rate = "Not directly logged per-node-existence; inferring from coverage column presence."
print("NOTE: the released CSVs log per-type *coverage* (whether the value was recovered in text),")
print("not a separate 'was a SENTIMENT node extracted at all' flag. The fact that")
print("emp_cov_sentiment / frm_cov_sentiment columns exist and are populated (not NaN) for every")
print("case confirms a SENTIMENT node IS constructed by Algorithm 1 for every case (the lexical")
print("classifier always returns a (label, confidence) pair per Eq. in Section IV-A) -- so the")
print("zero-rate is a GENERATION/REALIZATION failure (Stage 5, steered decoding never reproduces")
print("the sentiment label verbatim), not an EXTRACTION failure (Stage 2-3 always builds the node).")
print()
nan_check = combined[["emp_cov_sentiment", "frm_cov_sentiment"]].isna().mean()
print("Fraction of cases where the SENTIMENT coverage column is NaN (i.e. no node existed):")
print(nan_check)

KG structural stats (extraction-side) for SENTIMENT, vs URGENCY for contrast,
from the 600-case combined main-study data:

NOTE: the released CSVs log per-type *coverage* (whether the value was recovered in text),
not a separate 'was a SENTIMENT node extracted at all' flag. The fact that
emp_cov_sentiment / frm_cov_sentiment columns exist and are populated (not NaN) for every
case confirms a SENTIMENT node IS constructed by Algorithm 1 for every case (the lexical
classifier always returns a (label, confidence) pair per Eq. in Section IV-A) -- so the
zero-rate is a GENERATION/REALIZATION failure (Stage 5, steered decoding never reproduces
the sentiment label verbatim), not an EXTRACTION failure (Stage 2-3 always builds the node).

Fraction of cases where the SENTIMENT coverage column is NaN (i.e. no node existed):
emp_cov_sentiment    0.0
frm_cov_sentiment    0.0
dtype: float64


### Why this happens, and what it implies for "extend the ontology" (Section IX)

The diagnosis above shows the SENTIMENT node is **always constructed** (the lexical classifier in
Algorithm 1 always returns a label/confidence pair) — the failure is entirely downstream, at
**Stage 5 (steered generation)**: the model is never asked to literally repeat a sentiment label
like *"angry"* or *"polite"* back to the customer, because doing so would be conversationally
unnatural in an empathetic or formal support response. URGENCY, by contrast, recovers reasonably
well because urgency-bearing words ("urgent," "as soon as possible") are natural things to *say*
in a support reply, while a customer's own sentiment label is not something an agent would
typically restate verbatim.

This has a direct, useful implication for the "extending the ontology" future-work item: **the
near-invariance of KG structural statistics (Table II) is a property of the extraction pipeline,
but the *recoverability* of an entity type is a property of whether that type's surface value is
natural to restate in a generated response.** A richer ontology will likely reproduce this same
split — types that name concrete, restatable facts (order IDs, product names) should recover
well; types that are abstract scenario *labels* rather than literal content the agent would echo
(sentiment, perhaps category-style metadata fields) are structurally unlikely to recover under a
literal-substring rehydration check, regardless of ontology size. We recommend this be stated
explicitly in Section IX as a predicted, not just observed, limitation, and that the rehydration
operator (Eq. 9) be flagged as the natural place to fix it — e.g., by rehydrating SENTIMENT-type
nodes only into a structured metadata field rather than requiring the free-text response to
contain the literal label.

In [17]:
# --- Summary figure-ready table for a revised Fig. 6 / accompanying paragraph --------------
summary_table = (
    zero_infl.pivot_table(index="type", columns="style", values="zero_rate", aggfunc="mean")
    .round(3)
)
summary_table["overall_zero_rate"] = zero_infl.groupby("type")["zero_rate"].mean().round(3)
summary_table = summary_table.sort_values("overall_zero_rate", ascending=False)
print("Per-type zero-rate (mean across all 6 models), by style — drop-in for Section VII-G:")
summary_table

Per-type zero-rate (mean across all 6 models), by style — drop-in for Section VII-G:


style,emp,frm,overall_zero_rate
type,,,
SENTIMENT,1.000,0.998,0.999
CUSTOMER_NAME,0.993,0.997,0.995
ISSUE,0.993,0.990,0.992
ORDER_ID,0.985,0.951,0.968
PRODUCT,0.960,0.973,0.967
URGENCY,0.932,0.902,0.917


---
## 6. Manuscript-Ready Outputs

Consolidated drop-in text for the camera-ready, pulling directly from the computations above.

---
### Checklist mapping back to the review

- [x] **(1) Entity-coverage inconsistency** — root-caused to three different aggregations of the
  same per-case ratio; canonical equation + consistent-reporting fix proposed (Section 1).
- [x] **(2) Abstract/effect-magnitude tension** — exact absolute numbers computed; replacement
  abstract sentence drafted (Section 2).
- [~] **(3) Single-model ablation scope** — harness provided and tested for schema compatibility;
  **requires a live GPU run on your end** to produce the second data point (Section 3).
- [x] **(4) Response-length confound** — residualization + length-matched-subsample checks
  implemented and run on the released ablation data (Section 4).
- [x] **(5) SENTIMENT non-recovery** — promoted to a standalone, quantified, mechanistically
  explained analysis with a direct tie-in to the ontology-extension future-work item (Section 5).
